In [ ]:
# In this particular cell all the libraries are imported
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Data Loading**

In [ ]:
train_data = pd.read_csv('/kaggle/input/predict-the-success-of-bank-telemarketing/train.csv')
test_data = pd.read_csv('/kaggle/input/predict-the-success-of-bank-telemarketing/test.csv')
sample_data = pd.read_csv('/kaggle/input/predict-the-success-of-bank-telemarketing/sample_submission.csv')

# Creating Copies of Original Datasets

In [ ]:
train= train_data.copy()
test= test_data.copy()
sample= sample_data.copy()

# EDA (Exploratory Data Analysis)

# Data Preview

In [ ]:
# To know the shape of the dataset
train.shape    

In [ ]:
# To see numeric columns
import pandas as pd


numeric_columns = train.select_dtypes(include=['number'])


print(numeric_columns)


In [ ]:
# Lists all column names 

train.columns

In [ ]:
# To see the first 5 rows of the dataset
train.head()

In [ ]:

# Provides an overview of the dataset structure, including column names, data types, and non-null counts

train.info()

In [ ]:
 # Shows the count of missing values in each column 

train.isnull().sum()

In [ ]:
# Displays the number of unique values in each column

train.nunique()

# **Visualization of Numeric Columns**

In [ ]:
import matplotlib.pyplot as plt


train.hist(bins=10, figsize=(10, 8), color='blue', edgecolor='black')   # Creates histograms with 10 bins, blue color, and black edges for clarity
plt.suptitle('Histograms of Numerical Features', size=16)  # Adds a title for the entire figure
plt.show()   # Displays the plot


# Insights

 The histograms show that most features have a strong right-skew, meaning most values are low, with a few high outliers. Age is mainly between 20–60, and most people have low account balances, short call durations, and few contacts. Many zeros in pdays and previous suggest that most people were contacted for the first time in this campaign.

# Visualization of Categorical Columns

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a 3x2 grid of subplots (3 rows, 2 columns)
fig, axes = plt.subplots(3, 2, figsize=(14, 20))  # Adjust figsize as needed
plt.subplots_adjust(hspace=0.4)  # Adjust vertical space between plots

# Plot 1: Job distribution
sns.countplot(data=train, x='job', ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Job Types')
axes[0, 0].set_xlabel('Job')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# Plot 2: Marital status distribution
sns.countplot(data=train, x='marital', ax=axes[0, 1])
axes[0, 1].set_title('Distribution of Marital Status')
axes[0, 1].set_xlabel('Marital Status')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Education distribution
sns.countplot(data=train, x='education', ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Education Levels')
axes[1, 0].set_xlabel('Education Level')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: Default status distribution
sns.countplot(data=train, x='default', ax=axes[1, 1])
axes[1, 1].set_title('Distribution of Default Status')
axes[1, 1].set_xlabel('Default Status')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)

# Plot 5: Housing status distribution
sns.countplot(data=train, x='housing', ax=axes[2, 0])
axes[2, 0].set_title('Distribution of Housing Status')
axes[2, 0].set_xlabel('Housing Status')
axes[2, 0].set_ylabel('Count')
axes[2, 0].tick_params(axis='x', rotation=45)

# Plot 6: Loan status distribution
sns.countplot(data=train, x='loan', ax=axes[2, 1])
axes[2, 1].set_title('Distribution of Loan Status')
axes[2, 1].set_xlabel('Loan Status')
axes[2, 1].set_ylabel('Count')
axes[2, 1].tick_params(axis='x', rotation=45)

# Show the plots
plt.show()


# Insights


The data tells us that majority of the customers have blue collared jobs, are married, have secondary level education, are not defaulted, have housing loans and no personal loans.

# Data Cleaning

In [ ]:
 # Returns the count of duplicate (True) and unique (False) rows

train.duplicated().value_counts()

# Identifying Categorical Columns in the Dataset

In [ ]:
# To Select columns with categorical data types (object or category) and print their names

categorical_columns = train.select_dtypes(include=['object', 'category']).columns
print(categorical_columns)

# Detecting Null Values 

In [ ]:
# To Display the count of missing values in each column of the dataset

train.isnull().sum()

# Quick Insight 

In this dataset, only categorical columns (such as "job," "education," "contact," and "poutcome") have missing or null values, while all numerical columns have complete data with no missing values.

# Handling Null Values

In [ ]:
from sklearn.impute import SimpleImputer


categorical_imputer = SimpleImputer(strategy='most_frequent')

# List of categorical columns to impute
categorical_columns = ['job', 'education', 'contact', 'poutcome']

# Impute missing values for each categorical column
for column in categorical_columns:
    # Reshape the data and apply the imputer
    train[column] = categorical_imputer.fit_transform(train[[column]]).ravel()  # Use .ravel() to flatten the result

# Checking the imputed dataset
print(train[categorical_columns].isnull().sum())  # Should show 0 for all categorical columns


This approach is suitable for categorical data, as it preserves the distribution of categories without introducing new or arbitrary values, which helps maintain consistency of my data.

# Splitting the dataset into training and validation sets (80% train, 20% val)

In [ ]:
from sklearn.model_selection import train_test_split


X = train.drop('target', axis=1)  # Features
y = train['target']                 # Target variable


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # Split data into training and validation sets (80-20 split)


print(f"Training set shape: {X_train.shape}, Validation set shape: {X_val.shape}")
print(f"Training target shape: {y_train.shape}, Validation target shape: {y_val.shape}")


# Encoding and Scaling using Pipeline


In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# List of numeric columns to scale
columns_to_scale = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']

# Categorical columns for OneHotEncoder and OrdinalEncoder
categorical_columns_onehot = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'poutcome']
categorical_columns_ordinal = ['education']

# Drop the 'last contact date' column
X_train = X_train.drop(columns=['last contact date'])
X_val = X_val.drop(columns=['last contact date'])

# Create a column transformer to apply encoders and scalers
column_transformer = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(sparse_output=False), categorical_columns_onehot),
        ('ordinal', OrdinalEncoder(categories=[['primary', 'secondary', 'tertiary', 'unknown']]), categorical_columns_ordinal),
        ('scaler', StandardScaler(), columns_to_scale)
    ],
    remainder='passthrough',  # Keep the other columns unchanged
    verbose_feature_names_out=False  # Prevents verbose feature names
)

# Set the output to pandas DataFrame
column_transformer.set_output(transform='pandas')

# Create a pipeline that first transforms and then scales the data
pipeline = Pipeline(steps=[
    ('preprocessor', column_transformer)
])

# Fit and transform the training data
X_train_encoded_df = pipeline.fit_transform(X_train)

# Transform the validation data
X_val_encoded_df = pipeline.transform(X_val)

# Print the shapes of the resulting encoded and scaled datasets
print(f"Encoded & Scaled Training set shape: {X_train_encoded_df.shape}")
print(f"Encoded & Scaled Validation set shape: {X_val_encoded_df.shape}")

# Display the first few rows of the transformed datasets
print("Transformed Training Set:")
X_train_encoded_df.head()  # Shows the first 5 rows of the training set

print("\nTransformed Validation Set:")
X_val_encoded_df.head() # Shows the first 5 rows of the validation set


OneHotEncoding on columns like 'job,' 'marital,' and 'loan' as they are nominal categories without an inherent order

StandardScaler was chosen as it standardizes features by removing the mean and scaling to unit variance. This keeps each feature on a similar scale with a mean of 0 and a standard deviation of 1, which is ideal for many algorithms

In [ ]:
print("Transformed Training Set:")
X_train_encoded_df.head()

# Heatmap of Scaled Features Correlation

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Create a heatmap of the correlation matrix for the scaled training data
plt.figure(figsize=(14, 10))  # Adjust the size to fit the heatmap

# Compute the correlation matrix using the scaled values and original column names
correlation_matrix = pd.DataFrame(X_train_encoded_df, columns=X_train_encoded_df.columns).corr()

# Generate the heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)

# Set the title for the heatmap
plt.title('Heatmap of Scaled Features Correlation')

# Show the plot
plt.show()


# Insight

The heatmap reveals strong correlations between 'previous' and 'campaign', 'duration', and 'balance'. This suggests that frequent past contacts, longer interaction durations, and higher account balances are associated with increased customer engagement and potential future interactions.

# Dummy Submission

In [ ]:
df = pd.read_csv('/kaggle/input/predict-the-success-of-bank-telemarketing/train.csv')
X = df.drop('target', axis=1)
y = df['target']

from sklearn.dummy import DummyClassifier
model = DummyClassifier().fit(X, y)

X_test = pd.read_csv('/kaggle/input/predict-the-success-of-bank-telemarketing/test.csv')
y_pred = model.predict(X_test)

submission = pd.DataFrame({'id': range(0, X_test.shape[0]),
                           'target': y_pred})

submission.to_csv('submission.csv', index=False)

# Handling Null Values for test dataset

In [ ]:
# To see the first 5 rows of the test dataset

test.head()

# Checking for missing values on the test data

In [ ]:
# Counts the total number of missing (NaN) values in each column of the test dataset

test.isna().sum()

# Handling missing values on the test data

In [ ]:
from sklearn.impute import SimpleImputer


categorical_imputer = SimpleImputer(strategy='most_frequent')

# List of categorical columns to impute
categorical_columns = ['job', 'education', 'contact', 'poutcome']

# Impute missing values for each categorical column
for column in categorical_columns:
    # Reshape the data and apply the imputer
    test[column] = categorical_imputer.fit_transform(test[[column]]).ravel()  # Use .ravel() to flatten the result

# Optional: Check the imputed dataset
print(test[categorical_columns].isnull().sum())  # Should show 0 for all categorical columns


In [ ]:
# to view the shape of the test dataset

test.shape

In [ ]:
# To see the first 5 rows of the test dataset

test.head()

# Preprocessing for the test data

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Check if the target variable exists in the test dataset
print(test.columns)  # Print the columns to debug and confirm their names

# Create a column transformer to apply different encoders to different columns
column_transformer = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(sparse_output=False), ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'poutcome']),
        ('ordinal', OrdinalEncoder(categories=[['primary', 'secondary', 'tertiary', 'unknown']]), ['education'])
    ],
    remainder='passthrough',  # Keep the other columns unchanged
    verbose_feature_names_out=False  # Prevents verbose feature names
)

# Set output to pandas DataFrame
column_transformer.set_output(transform='pandas')

# Create a pipeline that first transforms the data
pipeline = Pipeline(steps=[
    ('preprocessor', column_transformer)
])

# Fit and transform the 'test' dataset
test_encoded_df = pipeline.fit_transform(test)

# Print the shape of the resulting encoded test dataset
print(f"Encoded Test set shape: {test_encoded_df.shape}")

# Display the first few rows of the transformed test dataset
print("Transformed Test Set:")
test_encoded_df.head() # Shows the first 5 rows of the test set


# Scaling for the test data

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# List of numeric columns you want to scale
columns_to_scale = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']

# Drop the 'last contact date' column from test_encoded_df
test_encoded_df = test_encoded_df.drop(columns=['last contact date'])

# Ensure all remaining columns are numeric before scaling
print(test_encoded_df.dtypes)  # This will show you the data types to confirm

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform only the specified columns in the test_encoded_df
test_encoded_df[columns_to_scale] = scaler.fit_transform(test_encoded_df[columns_to_scale])

# Print the shape of the resulting scaled test_encoded_df dataset
print(f"Scaled Test set shape: {test_encoded_df.shape}")

# Display the first few rows of the scaled dataset
print("Scaled Test Set:")
test_encoded_df.head()


In [ ]:

# retrieves and displays the column names of the DataFrame 'test_encoded_df'

test_encoded_df.columns

In [ ]:
# retrieves and displays the column names of the DataFrame 'X_train_encoded_df'

X_train_encoded_df.columns

# Models

# Model 1 (SVM with 'RBF' kernel)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

svm_rbf_clf = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_rbf_clf.fit(X_train_encoded_df, y_train)
y_val_pred_svm_rbf = svm_rbf_clf.predict(X_val_encoded_df)

accuracy_svm_rbf = accuracy_score(y_val, y_val_pred_svm_rbf)
f1_svm_rbf = f1_score(y_val, y_val_pred_svm_rbf, average='macro')
print(f"SVM RBF Validation Accuracy: {accuracy_svm_rbf:.4f}")
print(f"SVM RBF F1 Score: {f1_svm_rbf:.4f}")


# Model 2 (SVM with polynomial kernel)

In [ ]:
svm_poly_clf = SVC(kernel='poly', degree=3, class_weight='balanced', random_state=42)
svm_poly_clf.fit(X_train_encoded_df, y_train)
y_val_pred_svm_poly = svm_poly_clf.predict(X_val_encoded_df)

accuracy_svm_poly = accuracy_score(y_val, y_val_pred_svm_poly)
f1_svm_poly = f1_score(y_val, y_val_pred_svm_poly, average='macro')
print(f"SVM Polynomial Validation Accuracy: {accuracy_svm_poly:.4f}")
print(f"SVM Polynomial F1 Score: {f1_svm_poly:.4f}")


# Model 3 (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Initialize and fit the Logistic Regression model
log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train_encoded_df, y_train)

# Make predictions on the validation set
y_val_pred_log = log_reg.predict(X_val_encoded_df)

# Calculate accuracy and F1 score
accuracy_log = accuracy_score(y_val, y_val_pred_log)
f1_log = f1_score(y_val, y_val_pred_log, average='macro') # avaerage = macro means F1 score is calculated separately for each class.

# Print the results
print(f"Logistic Regression Validation Accuracy: {accuracy_log:.4f}")
print(f"Logistic Regression F1 Score: {f1_log:.4f}")


Logistic regression is used for binary classification problems. It predicts the probability of an outcome belonging to one of two classes. it is a simple yet effective alogirthm for classification problems, it maps any input value to a value of 0 or 1.

# Model 4 (SVM with Linear Kernel)

In [ ]:
# Import necessary libraries
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Initialize the Support Vector Classifier with class_weight parameter
svm_clf = SVC(kernel='linear', class_weight='balanced', random_state=42)

# Fit the model on the training data
svm_clf.fit(X_train_encoded_df, y_train)

# Make predictions on the validation data
y_val_pred_svm = svm_clf.predict(X_val_encoded_df)

In [ ]:
a= pd.Series(y_val_pred_svm)
type(a)
a.value_counts()

In [ ]:
# Evaluate the model
accuracy_svm = accuracy_score(y_val, a)

# Calculate F1 score
f1_svm = f1_score(y_val, a, average='macro')  # Use 'binary' for binary classification

# Output the results
print(f"SVM Validation Accuracy: {accuracy_svm:.4f}")
print(f"SVM F1 Score: {f1_svm:.4f}")

# Model Comparison based on F1 Score

In [ ]:
import matplotlib.pyplot as plt

# Model names
models = [
     "SVM (RBF Kernel)", 
    "SVM (Polynomial Kernel)", "Logistic Regression", "SVM (Linear Kernel)"
]

# F1 scores for each model
f1_scores = [ 0.7300, 0.7265, 0.7156, 0.7304]

# Plotting
plt.figure(figsize=(5, 3))
plt.barh(models, f1_scores, color="blue")
plt.xlabel("F1 Score")
plt.title("Model Comparison Based on F1 Score")
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()


# Insight

The SVM models with RBF and Linear kernels achieved the highest F1 scores, outperforming other models in this comparison, with the Linear kernel SVM reaching the top score of 0.7304. Overall, SVM variants and ensemble methods like XGBoost and Gradient Boost performed better than simpler models. Given its strong performance, I’ll proceed with the Linear kernel SVM model and apply hyperparameter tuning to further optimize it.

# Hyperparameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
import numpy as np

# Define the parameter grid with distributions to sample from
param_dist = {
    'C': np.logspace(-2, 2, 10),  # Randomly sample values of C
    'kernel': ['linear', 'rbf'],  # Kernel types
    'gamma': ['scale', 'auto'],   # Kernel coefficient (only for rbf)
    'class_weight': [None, 'balanced']
}

# Initialize the RandomizedSearchCV object
random_search = RandomizedSearchCV(estimator=SVC(random_state=42), 
                                   param_distributions=param_dist, 
                                   n_iter=20,  # Number of random combinations to try
                                   scoring='f1_macro',  # Use F1 score for tuning
                                   cv=5,  # 5-fold cross-validation
                                   verbose=2, 
                                   n_jobs=-1, 
                                   random_state=42)

# Fit RandomizedSearchCV on the training data
random_search.fit(X_train_encoded_df, y_train)

# Get the best parameters from the random search
best_params = random_search.best_params_
print(f"Best Parameters: {best_params}")

# Train the model with the best parameters on the entire training set
best_svm_clf = random_search.best_estimator_

# Make predictions on the validation set
y_val_pred_best_svm = best_svm_clf.predict(X_val_encoded_df)

# Evaluate the model
best_accuracy_svm = accuracy_score(y_val, y_val_pred_best_svm)
best_f1_svm = f1_score(y_val, y_val_pred_best_svm, average='macro')

# Output the results
print(f"Tuned SVM Validation Accuracy: {best_accuracy_svm:.4f}")
print(f"Tuned SVM F1 Score: {best_f1_svm:.4f}")




In [ ]:
y_val_pred_best_svm= best_svm_clf.predict(test_encoded_df)


In [ ]:
# Creating a DataFrame for submission with 'id' and 'target' columns, then saving it as 'submission.csv' without the index

submission = pd.DataFrame({'id': range(0, len(test_encoded_df)),
                           'target': y_val_pred_best_svm})

submission.to_csv('submission.csv', index=False)


In [ ]:

# Loading the 'submission.csv' file into a DataFrame and displaying the first 10 rows

submission = pd.read_csv('submission.csv')


submission.head(10)
